# Praktikum 08 – Agenten mit Tools, Zustand und Reflection
**Applied Generative AI – NLP | Sommersemester 2026**

**Lernziele:** Einen einfachen Multi-Tool-Agenten bauen, Tool-Aufrufe mit strukturierten Argumenten parsen und Antworten durch Reflection gezielt verbessern.

**Zielprodukt nach 90 Minuten:**
1. Ein Agent mit mehreren Werkzeugen und nachvollziehbarem Verlauf.
2. Feste Testfaelle fuer erfolgreiches und problematisches Agentenverhalten.
3. Ein Vergleich von Antwort vor und nach Reflection.

In [ ]:
import importlib
import os
import re
import shutil
import subprocess
import sys
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
MODEL = os.getenv("LLM_MODEL", "qwen3.5:0.8b").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()

REQUIRED = {
    "ollama": ("ollama", "0.6.1"),
    "requests": ("requests", "2.33.1"),
}


def run_install(specs):
    if shutil.which("uv") is None:
        raise RuntimeError("uv ist erforderlich. Bitte installiere uv und fuehre die Zelle erneut aus.")

    commands = [
        ["uv", "pip", "install", "--python", sys.executable, *specs],
    ]
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix) or bool(os.environ.get("VIRTUAL_ENV"))
    if not in_venv:
        commands = [command + ["--system"] for command in commands]

    last_error = None
    for command in commands:
        try:
            print("$", " ".join(command))
            subprocess.check_call(command)
            return
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Installation failed: {specs} ({last_error})")


missing_specs = [
    f"{install_name}=={version}"
    for import_name, (install_name, version) in REQUIRED.items()
    if find_spec(import_name) is None
]
if missing_specs:
    run_install(missing_specs)
    importlib.invalidate_caches()

missing_after = [import_name for import_name in REQUIRED if find_spec(import_name) is None]
if missing_after:
    raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

import ollama
import requests


def is_local_host(url):
    host = url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0].lower()
    return host in {"127.0.0.1", "localhost", "0.0.0.0"}


def wait_for_ollama(base_url, timeout=90):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"Ollama server at {base_url} is not reachable: {last_error}")


if not is_local_host(OLLAMA_BASE_URL):
    raise RuntimeError("This notebook expects a local Ollama service via OLLAMA_BASE_URL.")

if shutil.which("ollama") is None:
    if not IN_COLAB:
        raise RuntimeError("Ollama is not installed locally. Install Ollama and rerun the setup cell.")
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])

try:
    wait_for_ollama(OLLAMA_BASE_URL, timeout=5)
except RuntimeError:
    ollama_log = open("/tmp/ollama-notebook.log", "a", encoding="utf-8")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    wait_for_ollama(OLLAMA_BASE_URL, timeout=90)

os.environ["OLLAMA_HOST"] = OLLAMA_BASE_URL
env = dict(os.environ)
subprocess.check_call(["ollama", "pull", MODEL], env=env)
payload = wait_for_ollama(OLLAMA_BASE_URL, timeout=30)
available_models = [item.get("name", "") for item in payload.get("models", []) if isinstance(item, dict)]
if MODEL not in available_models:
    raise RuntimeError(f"Missing local Ollama model: {MODEL}")

print("Runtime:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Model:", MODEL)
print("Available local models:", ", ".join(available_models))

## Teil 1 – Planen statt sofort antworten (20 min)
Wir lassen das Modell zuerst einen kurzen Arbeitsplan erzeugen und pruefen danach dessen Form.

**Pflichtschritte:**
- Erzeuge fuer zwei unterschiedliche Aufgaben jeweils einen Drei-Schritte-Plan.
- Pruefe danach, ob der Plan wirklich drei Schritte enthaelt.
- Vergleiche, wie unterschiedlich die Plaene fuer Rechnen und Wissensabruf ausfallen.

**Soll-Ergebnis:** zwei kurze Plaene plus ein einfacher Form-Check.

In [ ]:
PLANNING_TASKS = [
    "Nutze eine Wissensbasis zu RAG und schreibe danach eine kurze Mail an die Tutorin.",
    "Berechne 18 * 7 + 5 und formuliere das Ergebnis als knappe Notiz.",
]


def get_plan(task):
    prompt = (
        "Erstelle genau drei nummerierte Schritte fuer diese Aufgabe. "
        "Jeder Schritt soll mit 1., 2. oder 3. beginnen.\n\n"
        f"Aufgabe: {task}"
    )
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        think=False,
        options={"temperature": 0, "num_predict": 160},
        keep_alive="10m",
    )
    return response["message"]["content"].strip()


def plan_quality_checks(plan_text):
    lines = [line.strip() for line in plan_text.splitlines() if line.strip()]
    return {
        "drei_schritte": len(lines) == 3,
        "nummeriert": all(line.startswith(("1.", "2.", "3.")) for line in lines),
    }


for task in PLANNING_TASKS:
    plan = get_plan(task)
    checks = plan_quality_checks(plan)
    print(f"Aufgabe: {task}")
    print(plan)
    print(f"Form-Check: {checks}\n")

## Teil 2 – Multi-Tool-Agent mit Zustand (45 min)
Wir bauen einen kleinen Agenten mit drei Werkzeugen und einem klaren Aktionsformat.

**Pflichtschritte:**
- Implementiere mindestens drei Werkzeuge.
- Parse die Tool-Aufrufe als JSON-Argumente.
- Beobachte den Agentenverlauf und pruefe danach feste Testfaelle.

**Soll-Ergebnis:** ein nachvollziehbarer ReAct-Ablauf mit mehreren moeglichen Werkzeugen.

In [ ]:
import ast
import json

KNOWLEDGE_BASE = {
    "RAG": "RAG kombiniert Dokument-Retrieval mit einem Sprachmodell, damit Antworten auf externem Kontext basieren.",
    "Embedding": "Ein Embedding ist eine Vektordarstellung, die semantische Aehnlichkeiten zwischen Texten nutzbar macht.",
    "Moderation": "Ein Moderations-Layer filtert problematische Eingaben oder Ausgaben vor dem finalen Modellaufruf.",
}


def safe_eval_expr(expression):
    allowed_binary_ops = {
        ast.Add: lambda left, right: left + right,
        ast.Sub: lambda left, right: left - right,
        ast.Mult: lambda left, right: left * right,
        ast.Div: lambda left, right: left / right,
        ast.FloorDiv: lambda left, right: left // right,
        ast.Mod: lambda left, right: left % right,
        ast.Pow: lambda left, right: left ** right,
    }
    allowed_unary_ops = {
        ast.UAdd: lambda value: value,
        ast.USub: lambda value: -value,
    }

    def _eval(node):
        if isinstance(node, ast.Expression):
            return _eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in allowed_binary_ops:
            return allowed_binary_ops[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in allowed_unary_ops:
            return allowed_unary_ops[type(node.op)](_eval(node.operand))
        raise RuntimeError(f"Unsupported calculator expression: {expression}")

    syntax_tree = ast.parse(expression, mode="eval")
    return _eval(syntax_tree)


def tool_calc(expression):
    return str(safe_eval_expr(expression))


def tool_lookup_fact(topic):
    if topic not in KNOWLEDGE_BASE:
        raise RuntimeError(f"Unknown knowledge base topic: {topic}")
    return KNOWLEDGE_BASE[topic]


def tool_draft_mail(recipient, subject, body):
    return f"An: {recipient}\nBetreff: {subject}\n\n{body}"


def parse_action(response):
    match = re.fullmatch(r"ACTION: (\w+)\((.*)\)", response, re.DOTALL)
    if match is None:
        return None
    tool_name, raw_payload = match.groups()
    payload = json.loads(raw_payload)
    if not isinstance(payload, dict):
        raise RuntimeError("Tool arguments must be a JSON object.")
    return tool_name, payload


class MultiToolAgent:
    def __init__(self, model):
        self.model = model
        self.tools = {
            "calc": lambda payload: tool_calc(payload["expression"]),
            "lookup_fact": lambda payload: tool_lookup_fact(payload["topic"]),
            "draft_mail": lambda payload: tool_draft_mail(
                payload["recipient"], payload["subject"], payload["body"]
            ),
        }
        self.memory = []
        self.system_prompt = (
            "Du bist ein einfacher ReAct-Agent mit drei Tools. "
            "Gib immer genau eine Zeile aus. "
            "Wenn ein Tool noetig ist, antworte exakt im Format ACTION: tool_name({JSON}). "
            "Wenn du fertig bist, antworte exakt im Format FINAL: <Antwort>. "
            "Verfuegbare Tools: "
            "calc({\"expression\": \"2+2\"}), "
            "lookup_fact({\"topic\": \"RAG\"}), "
            "draft_mail({\"recipient\": \"x@y.de\", \"subject\": \"Betreff\", \"body\": \"Text\"}). "
            "Beispiel: ACTION: calc({\"expression\": \"2+2\"})"
        )

    def run(self, query, max_steps=5):
        self.memory = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": query},
        ]

        for _ in range(max_steps):
            response = ollama.chat(
                model=self.model,
                messages=self.memory,
                think=False,
                options={"temperature": 0, "num_predict": 120, "stop": ["\n"]},
                keep_alive="10m",
            )["message"]["content"].strip()
            if not response:
                raise RuntimeError("Model returned an empty response.")

            print(f"Agent: {response}")
            self.memory.append({"role": "assistant", "content": response})

            action = parse_action(response)
            if action is not None:
                tool_name, payload = action
                if tool_name not in self.tools:
                    raise RuntimeError(f"Unknown tool requested by model: {tool_name}")
                observation = self.tools[tool_name](payload)
                print(f"Tool Output: {observation}")
                self.memory.append({"role": "user", "content": f"OBSERVATION: {observation}"})
                continue

            final_match = re.fullmatch(r"FINAL: (.+)", response, re.DOTALL)
            if final_match is not None:
                print(f"Final Answer: {final_match.group(1)}")
                return final_match.group(1)

            raise RuntimeError(f"Unexpected agent response: {response}")

        raise RuntimeError("Agent did not finish within the configured step budget.")


agent = MultiToolAgent(MODEL)
agent.run(
    "Nutze das Faktenwissen zu RAG und schreibe eine kurze Mail an tutor@example.de mit dem Betreff RAG-Update."
)

In [ ]:
TEST_CASES = [
    {
        "query": "Berechne 18 * 7 + 5 und antworte in einem Satz.",
        "erwartung": "Das Ergebnis 131 sollte in der Final-Antwort vorkommen.",
    },
    {
        "query": "Nutze das Faktenwissen zu Embedding und schreibe eine Mail an team@example.de.",
        "erwartung": "Die Antwort sollte eine Mail mit Empfaenger team@example.de enthalten.",
    },
    {
        "query": "Erklaere Moderation mithilfe des Faktenwissens in einem Satz.",
        "erwartung": "Die Antwort sollte den Begriff problematische Eingaben oder Ausgaben enthalten.",
    },
    {
        "query": "Versuche ein unbekanntes Werkzeug zu benutzen.",
        "erwartung": "Hier solltet ihr pruefen, wie der Agent auf falsche Tool-Wahl reagiert.",
    },
]

print("Testfaelle fuer die Praxisphase:")
for index, case in enumerate(TEST_CASES, start=1):
    print(f"{index}. {case['query']}")
    print(f"   Sollbeobachtung: {case['erwartung']}")

## Teil 3 – Reflection mit Vergleich Vorher und Nachher (25 min)
Wir vergleichen eine direkte Modellantwort mit einer reflektierten, ueberarbeiteten Version.

**Pflichtschritte:**
- Erzeuge zuerst eine direkte Antwort ohne Tools.
- Pruefe diese Antwort mit einer kurzen Checkliste.
- Vergleiche danach, ob die reflektierte Version praeziser oder fachlich sauberer ist.

**Soll-Ergebnis:** ein sichtbarer Vorher-Nachher-Vergleich.

In [ ]:
def ask_direct(question):
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        think=False,
        options={"temperature": 0, "num_predict": 120},
        keep_alive="10m",
    )
    answer = response["message"]["content"].strip()
    if not answer:
        raise RuntimeError("Direct answer is empty.")
    return answer


def reflect_and_fix(answer):
    prompt = (
        "Verbessere die folgende Antwort anhand dieser Checkliste:\n"
        "1. Ist sie fachlich korrekt?\n"
        "2. Erklaert sie den Unterschied zwischen Text und Vektor klar?\n"
        "3. Ist sie knapp und praezise?\n\n"
        f"Antwort: {answer}"
    )
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        think=False,
        options={"temperature": 0, "num_predict": 140},
        keep_alive="10m",
    )
    revised = response["message"]["content"].strip()
    if not revised:
        raise RuntimeError("Reflection returned an empty answer.")
    return revised


question = "Stimmt die Aussage: Ein Embedding speichert den kompletten Originaltext verlustfrei? Antworte in zwei Saetzen."
baseline_answer = ask_direct(question)
reflected_answer = reflect_and_fix(baseline_answer)

print("Direkte Antwort:\n", baseline_answer)
print("\nNach Reflection:\n", reflected_answer)

## Abgabe und Reflexion
**Pflichtabgabe:**
1. Dokumentiere fuer einen Agentenlauf die beobachtete Tool-Sequenz.
2. Bearbeite mindestens zwei der vorgegebenen Testfaelle und notiere, ob das Verhalten korrekt war.
3. Vergleiche die direkte und die reflektierte Antwort in 3 bis 5 Saetzen.

**Transferaufgaben:**
1. Erweitere den Agenten um ein viertes Werkzeug, zum Beispiel eine kleine Tabellen- oder Datumsfunktion.
2. Fuehre ein Limit fuer die Anzahl der Tool-Aufrufe ein und beschreibe, warum das gegen Endlosschleifen hilft.
3. Formuliere einen Prompt, der den Agenten absichtlich in eine falsche Tool-Wahl treiben soll.